# EDA with Data Visualization

Scatter plots compare flight number and payload with sites and orbits. Orbit bars include sample counts. Yearly averages describe a changing technology and mission mix.

Prepared with AI assistance. Data and outputs are reproducible.

In [1]:
"""Reproducible IBM Applied Data Science Capstone analysis.
Run: python analysis.py. Historical course snapshots are included in data/.
"""
from pathlib import Path
import json, sqlite3, io, contextlib, base64
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import folium
from folium.plugins import MarkerCluster, MousePosition
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, balanced_accuracy_score
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
DATA=ROOT/'data'; RESULTS=ROOT/'results'; RESULTS.mkdir(exist_ok=True)
plt.rcParams.update({'font.size':12,'axes.spines.top':False,'axes.spines.right':False})


def wrangle():
    df=pd.read_csv(DATA/'dataset_part_1.csv')
    bad={'False ASDS','False Ocean','False RTLS','None ASDS','None None'}
    df['Class']=(~df.Outcome.isin(bad)).astype(int)
    missing=df.isna().sum(); df['PayloadMass']=df.PayloadMass.fillna(df.PayloadMass.mean())
    assert len(df)==90 and df.FlightNumber.is_unique
    assert np.array_equal(df.Class,pd.read_csv(DATA/'dataset_part_2.csv').Class)
    df.to_csv(DATA/'analysis_dataset.csv',index=False)
    print('Published course API-derived checkpoint: 90 Falcon 9 launches')
    print('Date range:',df.Date.min(),'to',df.Date.max())
    print('Missing values before processing:\n',missing[missing>0].to_string())
    print('Outcome counts:\n',df.Outcome.value_counts().to_string())
    print('Class counts:',df.Class.value_counts().to_dict())
    return df


def saveplot(name):
    plt.tight_layout(); plt.savefig(RESULTS/(name+'.png'),dpi=170,bbox_inches='tight'); plt.close()


def eda(df):
    specs=[('FlightNumber','LaunchSite','flight_site'),('PayloadMass','LaunchSite','payload_site'),
           ('FlightNumber','Orbit','flight_orbit'),('PayloadMass','Orbit','payload_orbit')]
    for x,y,name in specs:
        fig,ax=plt.subplots(figsize=(9,4.4))
        for label,color in [(0,'#d97732'),(1,'#087f8c')]:
            q=df[df.Class==label]; ax.scatter(q[x],q[y],s=45,alpha=.78,c=color,label='Failure / no recovery' if label==0 else 'Success')
        ax.set_xlabel('Payload mass (kg)' if x=='PayloadMass' else 'Flight number'); ax.set_ylabel('Launch site' if y=='LaunchSite' else 'Orbit')
        ax.legend(loc='best',fontsize=10); saveplot(name)
    orbit=df.groupby('Orbit').Class.agg(['mean','count','sum']).sort_values('mean')
    fig,ax=plt.subplots(figsize=(9,4.4)); ax.bar(orbit.index,orbit['mean']*100,color='#087f8c')
    for i,(_,r) in enumerate(orbit.iterrows()): ax.text(i,r['mean']*100+2,f"n={int(r['count'])}",ha='center',fontsize=10)
    ax.set_ylim(0,115); ax.set_ylabel('Landing success (%)'); saveplot('orbit_rate')
    yearly=df.assign(Year=pd.to_datetime(df.Date).dt.year).groupby('Year').Class.agg(['mean','count','sum'])
    fig,ax=plt.subplots(figsize=(9,4.4)); ax.plot(yearly.index,yearly['mean']*100,'o-',color='#087f8c',linewidth=2)
    ax.set_ylim(0,105); ax.set_ylabel('Landing success (%)'); ax.set_xlabel('Year'); ax.grid(alpha=.2); saveplot('year_rate')
    site=df.groupby('LaunchSite').Class.agg(['mean','count','sum'])
    for name,table in [('orbit',orbit),('year',yearly),('site',site)]: table.to_csv(RESULTS/(name+'_summary.csv'))
    print('Launch-site success:\n',site.to_string()); print('Orbit success:\n',orbit.to_string()); print('Yearly success:\n',yearly.to_string())
    return site,orbit,yearly


In [2]:
df=wrangle()
site, orbit, yearly = eda(df)

Published course API-derived checkpoint: 90 Falcon 9 launches
Date range: 2010-06-04 to 2020-11-05
Missing values before processing:
 LandingPad    26
Outcome counts:
 Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Class counts: {1: 60, 0: 30}
Launch-site success:
                   mean  count  sum
LaunchSite                        
CCAFS SLC 40  0.600000     55   33
KSC LC 39A    0.772727     22   17
VAFB SLC 4E   0.769231     13   10
Orbit success:
            mean  count  sum
Orbit                      
SO     0.000000      1    0
GTO    0.518519     27   14
ISS    0.619048     21   13
MEO    0.666667      3    2
PO     0.666667      9    6
LEO    0.714286      7    5
VLEO   0.857143     14   12
HEO    1.000000      1    1
GEO    1.000000      1    1
ES-L1  1.000000      1    1
SSO    1.000000      5    5
Yearly success:
           mean  count  sum
Year                      
201